# LightGBM — Consommation électrique par usage (dataset complet)
**Périmètre :** tout le parc (549 971 logements, **aucun filtre**)
**Cibles :** électricité totale + par usage (chauffage, clim, eau chaude, prises, éclairage, frigo, cuisson, sèche-linge)
**Split :** stratification par zones ASHRAE IECC 2004
**Différence avec `lgbm_electricity` :** pas de `mask`, et on prédit plusieurs usages au lieu de la seule conso totale.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED      = 42
LGBM_BASE = dict(random_state=SEED, n_jobs=-1, verbose=-1)
# GPU : CUDA non compilé dans ce build ; device='gpu' (OpenCL) instable sur la T400 4 Go.
# -> on reste en CPU (dataset moyen, 4 cibles, gain GPU marginal ici).

# cibles électricité par usage (nom court -> colonne brute)
TARGETS = {
    'total'      : 'out.electricity.total.energy_consumption..kwh',
    'chauffage'  : 'out.electricity.heating.energy_consumption..kwh',
    'clim'       : 'out.electricity.cooling.energy_consumption..kwh',
    'eau_chaude' : 'out.electricity.hot_water.energy_consumption..kwh',
}

def to_numpy_dtypes(df):
    for col in df.columns:
        dt = df[col].dtype
        if hasattr(dt, 'numpy_dtype'):
            df[col] = df[col].astype(dt.numpy_dtype)
        elif hasattr(dt, 'pyarrow_dtype'):
            df[col] = df[col].astype(str(dt.pyarrow_dtype))
    return df

def metrics(y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

In [ ]:
ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'

# X complet (aucun filtre)
X = to_numpy_dtypes(pd.read_parquet(DATA_PROCESSED / 'X.parquet'))

# cibles par usage : lues dans le fichier brut (pas dans Y.parquet)
Yt = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=list(TARGETS.values()))
Yt = Yt.rename(columns={v: k for k, v in TARGETS.items()}).reset_index(drop=True)
Yt = to_numpy_dtypes(Yt)

# zone climatique pour la stratification
ashrae = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet',
                         columns=['in.ashrae_iecc_climate_zone_2004'])            ['in.ashrae_iecc_climate_zone_2004'].astype(str).reset_index(drop=True)

print(f'{len(X):,} logements | {X.shape[1]} features | {len(TARGETS)} cibles | {ashrae.nunique()} zones ASHRAE')
print('\nPart de logements avec conso > 0 par usage :')
for k in TARGETS:
    print(f'  {k:12} : {(Yt[k] > 0).mean()*100:5.1f}%  (moyenne {Yt[k].mean():.0f} kWh)')

In [ ]:
# Split unique (stratifié ASHRAE), partagé par toutes les cibles
counts_z = ashrae.value_counts()
rare_z   = counts_z[counts_z < 5].index
strat    = ashrae.where(~ashrae.isin(rare_z), other='RARE')

idx = np.arange(len(X))
idx_tv, idx_test = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=strat)
idx_train, idx_val = train_test_split(idx_tv, test_size=0.2, random_state=SEED,
                                      stratify=strat.iloc[idx_tv])

X_trainval, X_test = X.iloc[idx_tv], X.iloc[idx_test]
X_train,    X_val  = X.iloc[idx_train], X.iloc[idx_val]
ash_test           = ashrae.iloc[idx_test].values

print(f'Train : {len(idx_train):,} | Val : {len(idx_val):,} | Test : {len(idx_test):,}')

In [ ]:
# Optuna : optimisation des hyperparamètres sur la cible 'total'
Y_train_t = Yt['total'].iloc[idx_train]
Y_val_t   = Yt['total'].iloc[idx_val]

def objective(trial):
    params = {
        'n_estimators'     : 2000,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 120),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 400),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.1, 50.0, log=True),
        'reg_alpha'        : trial.suggest_float('reg_alpha',  0.01, 10.0, log=True),
        **LGBM_BASE,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train, Y_train_t, eval_set=[(X_val, Y_val_t)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return np.sqrt(mean_squared_error(Y_val_t, m.predict(X_val)))

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=25, show_progress_bar=True)   # ↑ augmenter si besoin

print(f'Best val RMSE (total) : {study.best_value:.0f} kWh')
best_params = {**study.best_params, 'n_estimators': 5000, **LGBM_BASE}
best_params

In [ ]:
# Entraînement d'un modèle par usage (mêmes hyperparamètres) + collecte des métriques
resultats = {}
preds_test = {}
models = {}

for nom, col in TARGETS.items():
    y_tv = Yt[nom].iloc[idx_tv]
    y_te = Yt[nom].iloc[idx_test]

    m = lgb.LGBMRegressor(**best_params)
    m.fit(X_trainval, y_tv, eval_set=[(X_test, y_te)],
          callbacks=[lgb.early_stopping(100, verbose=False)])

    p = m.predict(X_test)
    r2, rmse, mae = metrics(y_te, p)
    moy = y_te.mean()
    resultats[nom] = {
        'R2': r2, 'RMSE': rmse, 'MAE': mae,
        'RMSE_%': rmse / moy * 100 if moy > 0 else np.nan,
        'moyenne': moy, 'part_>0_%': (y_te > 0).mean() * 100,
        'arbres': m.best_iteration_,
    }
    preds_test[nom] = p
    models[nom] = m
    print(f'{nom:12} | R²={r2:.4f} | RMSE={rmse:7.0f} kWh | MAE={mae:6.0f} | arbres={m.best_iteration_}')

res = pd.DataFrame(resultats).T
res = res[['R2', 'RMSE', 'RMSE_%', 'MAE', 'moyenne', 'part_>0_%', 'arbres']].round(2)
print('\n=== Résultats par usage (test) ===')
print(res.to_string())

In [ ]:
# Graphiques comparatifs par usage
ordre = res.sort_values('R2', ascending=True).index

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(ordre, res.loc[ordre, 'R2'], color='steelblue', edgecolor='white')
axes[0].set(title='R² par usage', xlabel='R² (test)'); axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(res.loc[ordre, 'R2']):
    axes[0].text(v, i, f' {v:.3f}', va='center', fontsize=9)

axes[1].barh(ordre, res.loc[ordre, 'RMSE'], color='coral', edgecolor='white')
axes[1].set(title='RMSE par usage', xlabel='RMSE (kWh)'); axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(res.loc[ordre, 'RMSE']):
    axes[1].text(v, i, f' {v:.0f}', va='center', fontsize=9)

plt.suptitle('Performance LightGBM par usage électrique — dataset complet', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Réel vs Prédit pour chaque usage
n = len(TARGETS)
ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 4.2*nrow))
axes = axes.ravel()

for ax, nom in zip(axes, TARGETS):
    yt = Yt[nom].iloc[idx_test].values
    p  = preds_test[nom]
    ax.scatter(yt, p, s=4, alpha=0.2, color='steelblue', edgecolors='none')
    hi = max(yt.max(), p.max()) * 1.02
    ax.plot([0, hi], [0, hi], 'k--', lw=1)
    ax.text(0.04, 0.95, f"R²={resultats[nom]['R2']:.3f}", transform=ax.transAxes,
            va='top', fontsize=11, bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    ax.set(title=nom, xlabel='réel (kWh)', ylabel='prédit (kWh)')

for ax in axes[n:]:
    ax.axis('off')
plt.suptitle('Réel vs Prédit par usage électrique', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Importance des features pour un usage choisi (ex. chauffage)
usage = 'chauffage'   # <-- changer pour explorer un autre usage
TOP_N = 20            # nombre de features affichées

imp = pd.Series(models[usage].feature_importances_,
                index=X_trainval.columns).sort_values(ascending=False)
top = imp.head(TOP_N).sort_values()

# noms tronqués pour rester lisibles
labels = [name if len(name) <= 45 else name[:42] + '…' for name in top.index]

fig, ax = plt.subplots(figsize=(11, 0.45 * TOP_N + 1))
ax.barh(range(len(top)), top.values, color='seagreen')
ax.set_yticks(range(len(top)))
ax.set_yticklabels(labels, fontsize=9)
ax.set(title=f'Feature importance — usage « {usage} » (top {TOP_N})', xlabel='Importance')
ax.grid(axis='x', alpha=0.3)
ax.margins(y=0.01)
plt.tight_layout()
plt.show()

print(f'Top 15 features pour « {usage} » :')
print(imp.head(15).round(0).to_string())
